# Build sum-of-products KEO for water ($\mathrm{H_2O}$) in valence coordinates

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from keosop.gmat import Gmat
from keosop.keo_jax import batch_Gmat, com
from keosop.revad import dot, norm

jax.config.update("jax_enable_x64", True)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{H_2O}$.

The internal coordinates are defined as:

1. $r_{\mathrm{OH_1}}$ — O-H1 bond length,
2. $r_{\mathrm{OH_2}}$ — O-H2 bond length,
3. $\alpha_{\mathrm{H_1OH_2}}$ — H1–O-H2 bond angle,

The atom ordering in Cartesian coordinates is:

$$
\mathrm{O},\ \mathrm{H_1},\ \mathrm{H_2}
$$

In [2]:
masses = np.array([15.99491463, 1.00782505, 1.00782505])  # masses of O, H, H

ncoo = 3


def internal_to_cartesian(q):
    r1, r2, a = q
    return np.array(
        [
            [0.0, 0.0, 0.0],
            [r1 * np.sin(a / 2), 0.0, r1 * np.cos(a / 2)],
            [-r2 * np.sin(a / 2), 0.0, r2 * np.cos(a / 2)],
        ],
        dtype=object,
    )


def cartesian_to_internal(xyz):
    C, H1, H2 = xyz

    CH1 = H1 - C
    CH2 = H2 - C

    r1 = norm(CH1)
    r2 = norm(CH2)
    a = np.arccos(dot(CH1, CH2) / (r1 * r2))

    return np.array([r1, r2, a], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the G-matrix in sum-of-products form

In [3]:
grids = [
    np.linspace(0.8, 2.0, 10),  # r1
    np.linspace(0.8, 2.0, 10),  # r2
    np.linspace(30 * np.pi / 180, 170 * np.pi / 180, 10),  # alpha
]

gmat = Gmat(grids, masses, internal_to_cartesian, cartesian_to_internal)
gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

print("Number of terms in Gvib_{i,j}\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print("\nNumber of terms in Grovib_{i,j}\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print("\nNumber of terms in Grot_{i,j}\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

Number of terms in Gvib_{i,j}
 i  j no.terms
 0  0      1
 0  1      2
 0  2      3
 1  1      1
 1  2      6
 2  2     12

Number of terms in Grovib_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 1  0      2
 1  1      2
 1  2     13
 2  0      1
 2  1      1
 2  2      1

Number of terms in Grot_{i,j}
 i  j no.terms
 0  0      3
 0  1      1
 0  2      2
 1  1      7
 1  2      1
 2  2      3


Evaluate the G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [4]:
@com(masses)
def internal_to_cartesian_jax(q):
    r1, r2, a = q
    return jnp.array(
        [
            [0.0, 0.0, 0.0],
            [r1 * jnp.sin(a / 2), 0.0, r1 * jnp.cos(a / 2)],
            [-r2 * jnp.sin(a / 2), 0.0, r2 * jnp.cos(a / 2)],
        ],
        dtype=jnp.float64,
    )


grid_ndim = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
grid_ndim = grid_ndim.reshape(-1, len(grids))

gmat = batch_Gmat(grid_ndim, masses, internal_to_cartesian_jax)

gvib_ndim = gmat[:, :ncoo, :ncoo]
grovib_ndim = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_ndim = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

Calculate point-wise differences between the sum-of-products and numerical G-matrix

In [5]:
# number of multidimensional grid points
ngrid = grid_ndim.shape[0]

# evaluate the sum-of-products G-matrix on the full direct-product grid
gvib_sop_eval = np.zeros((ngrid, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((ngrid, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((ngrid, 3, ncoo), dtype=np.float64)

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

# Gvib
for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            sop = gvib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            gvib_sop_eval[ipoint, i, j] = value * G_to_invcm

# Grot
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            sop = grot_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grot_sop_eval[ipoint, i, j] = value * G_to_invcm

# Grovib
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            sop = grovib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            grovib_sop_eval[ipoint, i, j] = value * G_to_invcm

In [6]:
# differences

diff_vib = gvib_sop_eval - gvib_ndim
diff_rovib = grovib_sop_eval - grovib_ndim
diff_rot = grot_sop_eval - grot_ndim

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)

print("--- Gvib ---")
print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\n--- Grovib ---")
print("Maximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\n--- Grot ---")
print("Maximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

--- Gvib ---
Maximum absolute difference: 1.0942358130705543e-12
Mean absolute difference: 1.0205057478075553e-14
RMS difference: 4.927411161027335e-14

--- Grovib ---
Maximum absolute difference: 6.920144109185082e-09
Mean absolute difference: 4.840044430353093e-10
RMS difference: 9.949773408397386e-10

--- Grot ---
Maximum absolute difference: 3.0645583848354363e-09
Mean absolute difference: 6.679511882948778e-11
RMS difference: 2.9345334999112687e-10


In [7]:
print("--- Gvib element-wise errors ---")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff_vib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Gvib[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

print("\n--- Grovib element-wise errors ---")

for i in range(3):
    for j in range(ncoo):
        d = diff_rovib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Grovib[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

print("\n--- Grot element-wise errors ---")

for i in range(3):
    for j in range(i, 3):
        d = diff_rot[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"Grot[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

--- Gvib element-wise errors ---
Gvib[0,0]  max abs = 2.132e-14   RMS = 1.421e-14   
Gvib[0,1]  max abs = 7.772e-16   RMS = 3.469e-16   
Gvib[0,2]  max abs = 8.438e-15   RMS = 1.861e-15   
Gvib[1,1]  max abs = 2.132e-14   RMS = 9.662e-15   
Gvib[1,2]  max abs = 3.930e-14   RMS = 1.186e-14   
Gvib[2,2]  max abs = 1.094e-12   RMS = 1.458e-13   

--- Grovib element-wise errors ---
Grovib[0,0]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[0,1]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[0,2]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[1,0]  max abs = 2.983e-09   RMS = 1.429e-09   
Grovib[1,1]  max abs = 2.960e-09   RMS = 1.403e-09   
Grovib[1,2]  max abs = 6.920e-09   RMS = 2.213e-09   
Grovib[2,0]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[2,1]  max abs = 0.000e+00   RMS = 0.000e+00   
Grovib[2,2]  max abs = 0.000e+00   RMS = 0.000e+00   

--- Grot element-wise errors ---
Grot[0,0]  max abs = 2.728e-12   RMS = 3.245e-13   
Grot[0,1]  max abs = 0.000e+00   RMS = 0.000e+0

In [8]:
print("--- Gvib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_vib), abs_diff_vib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", gvib_sop_eval[ipoint, i, j])
print("reference  :", gvib_ndim[ipoint, i, j])
print("difference :", diff_vib[ipoint, i, j])

print("\n--- Grovib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rovib), abs_diff_rovib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", grovib_sop_eval[ipoint, i, j])
print("reference  :", grovib_ndim[ipoint, i, j])
print("difference :", diff_rovib[ipoint, i, j])

print("\n--- Grot largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rot), abs_diff_rot.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(grid_ndim[ipoint]))
print("element    :", (i, j))
print("SOP        :", grot_sop_eval[ipoint, i, j])
print("reference  :", grot_ndim[ipoint, i, j])
print("difference :", diff_rot[ipoint, i, j])

--- Gvib largest discrepancy ---
grid point : 19
coordinates: [0.8        0.93333333 2.96705973]
element    : (np.int64(2), np.int64(2))
SOP        : 101.94791933929002
reference  : 101.94791933929112
difference : -1.0942358130705543e-12

--- Grovib largest discrepancy ---
grid point : 9
coordinates: [0.8        0.8        2.96705973]
element    : (np.int64(1), np.int64(2))
SOP        : 6.9201438706933404e-09
reference  : -2.3849174140324253e-16
difference : 6.920144109185082e-09

--- Grot largest discrepancy ---
grid point : 99
coordinates: [0.8        2.         2.96705973]
element    : (np.int64(1), np.int64(1))
SOP        : 15.46503659329558
reference  : 15.465036590231021
difference : 3.0645583848354363e-09
